In [1]:
import pandas as pd
import numpy as np

# Load cleaned FX data
df = pd.read_csv("../data/processed/cleaned_fx_data.csv")

# Prepare Risk Engine base (do not assume Date column exists)
risk_df = df.copy()

# Safety check: ensure volatility column exists
assert "USD_Volatility" in risk_df.columns, "USD_Volatility column not found in dataset"

# Simulated exposure size (Aadhithya's business input)
risk_df["Exposure_USD"] = np.random.randint(50000, 500000, size=len(risk_df))

risk_df.head()


,Unnamed: 0,EUR,GBP,JPY,USD,USD_Return,USD_Volatility,GBP_Return,GBP_Volatility,EUR_Return,EUR_Volatility,JPY_Return,JPY_Volatility,Exposure_USD
0,2016-01-01,71.8627,97.6059,55.01,66.1780,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,446047
1,2016-01-02,71.8627,97.6059,55.01,66.1780,0.000000,NaN,0.000000,NaN,0.000000,NaN,0.000000,NaN,184382
2,2016-01-03,71.8627,97.6059,55.01,66.1780,0.000000,NaN,0.000000,NaN,0.000000,NaN,0.000000,NaN,82379
3,2016-01-04,72.3907,97.9455,55.65,66.4623,0.004296,NaN,0.003479,NaN,0.007347,NaN,0.011634,NaN,107436
4,2016-01-05,72.0315,97.9562,55.71,66.5418,0.001196,NaN,0.000109,NaN,-0.004962,NaN,0.001078,NaN,296362


In [2]:
# Normalize to 0–100
normalize = lambda x: ((x - x.min()) / (x.max() - x.min())) * 100

risk_df["Volatility_Score"] = normalize(risk_df["USD_Volatility"])
risk_df["Exposure_Score"] = normalize(risk_df["Exposure_USD"])

# Risk Score (Weighted)
risk_df["Risk_Score"] = (
    0.6 * risk_df["Volatility_Score"] +
    0.4 * risk_df["Exposure_Score"]
)

# Risk Levels
risk_df["Risk_Level"] = risk_df["Risk_Score"].apply(
    lambda x: "Low" if x < 40 else "Medium" if x <= 70 else "High"
)

risk_df[["Risk_Score", "Risk_Level"]].head()


,Risk_Score,Risk_Level
0,NaN,High
1,NaN,High
2,NaN,High
3,NaN,High
4,NaN,High


In [3]:
# Risk Alert Logic
risk_df["Risk_Alert"] = risk_df["Risk_Level"].apply(
    lambda x: "🚨 HIGH RISK ALERT" if x == "High" else "No Alert"
)

# Top 10% Volatility → CRITICAL RISK
critical_threshold = risk_df["USD_Volatility"].quantile(0.90)
risk_df["Critical_Risk_Flag"] = risk_df["USD_Volatility"].apply(
    lambda x: "CRITICAL RISK" if x >= critical_threshold else "Normal"
)

print("Risk Levels:\n", risk_df["Risk_Level"].value_counts())
print("\nCritical Risk Days:\n", risk_df["Critical_Risk_Flag"].value_counts())

risk_df.tail()


Risk Levels:
 Medium    1801
Low       1660
High       193
Name: Risk_Level, dtype: int64

Critical Risk Days:
 Normal           3291
CRITICAL RISK     363
Name: Critical_Risk_Flag, dtype: int64


,Unnamed: 0,EUR,GBP,JPY,USD,USD_Return,USD_Volatility,GBP_Return,GBP_Volatility,EUR_Return,EUR_Volatility,JPY_Return,JPY_Volatility,Exposure_USD,Volatility_Score,Exposure_Score,Risk_Score,Risk_Level,Risk_Alert,Critical_Risk_Flag
3649,2025-12-28,105.8522,121.2400,57.50,89.8296,0.000000,0.002839,0.000000,0.003616,0.000000,0.004022,0.000000,0.005377,218861,48.606924,37.474646,44.154013,Medium,No Alert,Normal
3650,2025-12-29,105.8407,121.4053,57.58,89.9756,0.001625,0.002852,0.001363,0.003614,-0.000109,0.004023,0.001391,0.005381,163069,48.840661,25.066276,39.330907,Low,No Alert,Normal
3651,2025-12-30,105.9269,121.5656,57.64,89.9429,-0.000363,0.002853,0.001320,0.003611,0.000814,0.004021,0.001042,0.005383,133315,48.870904,18.448865,36.702088,Low,No Alert,Normal
3652,2025-12-31,105.5557,121.0237,57.42,89.9198,-0.000257,0.002796,-0.004458,0.003717,-0.003504,0.004044,-0.003817,0.005235,94662,47.810401,9.852279,32.627152,Low,No Alert,Normal
3653,2026-01-01,105.6915,121.2476,57.42,89.9792,0.000661,0.002783,0.001850,0.003722,0.001287,0.004021,0.000000,0.005227,380240,47.573459,73.365997,57.890474,Medium,No Alert,Normal


In [4]:
# 1) Risk Trend (7-day rolling average)
risk_df["Risk_Trend_7D"] = risk_df["Risk_Score"].rolling(window=7).mean()

# 2) Exposure Sensitivity Check
risk_df["Exposure_Level"] = risk_df["Exposure_USD"].apply(
    lambda x: "High Exposure" if x > risk_df["Exposure_USD"].median() else "Low Exposure"
)

risk_df.groupby("Exposure_Level")["Risk_Score"].mean()


Exposure_Level
High Exposure    52.213326
Low Exposure     31.749727
Name: Risk_Score, dtype: float64

In [5]:
# Action Recommendation (create FIRST)
risk_df["Action_Recommendation"] = risk_df["Risk_Level"].apply(
    lambda x: "Immediate Hedging" if x == "High"
    else "Monitor Closely" if x == "Medium"
    else "No Action"
)

# Top 5 highest risk days (extract AFTER all columns exist)
top_risk_days = risk_df.sort_values(by="Risk_Score", ascending=False).head(5)

top_risk_days[
    ["USD_Volatility", "Exposure_USD", "Risk_Score", "Risk_Level", "Action_Recommendation"]
]


,USD_Volatility,Exposure_USD,Risk_Score,Risk_Level,Action_Recommendation
1091,0.005476,470347,95.747205,High,Immediate Hedging
1086,0.005507,465149,95.622384,High,Immediate Hedging
1084,0.005486,454368,94.436029,High,Immediate Hedging
1096,0.005126,493838,93.959183,High,Immediate Hedging
1085,0.005513,425137,92.133019,High,Immediate Hedging
